In [ ]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from joblib import dump

# ---- paths ----
base = Path("../datasets")
agent_type = "random_prompt"
train_set_path = base / f"{agent_type}_train.npz"
test_set_path = base / f"{agent_type}_test.npz"
model_path = base / f"{agent_type}_train_rf.joblib"

In [ ]:
## Loading the training dataset and check the data

d = np.load(Path(train_set_path))
X, y = d["dataset"], d["label"]

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("classes:", np.unique(y), "counts:", np.bincount(y.astype(int)))
print("sample[0] shape:", X[0].shape)
print("sample[0] first 10:", X[0].reshape(-1)[:10])
labels = np.unique(y)
print("Unique labels:", labels)

In [ ]:
## Random Forest code

# ---- load ----
d = np.load(train_set_path)
X = d["dataset"]
y = d["label"]

print("Loaded:", train_set_path)
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)

# ---- flatten for RF: [N, ...] -> [N, D] ----
X2 = X.reshape(X.shape[0], -1)
y = y.astype(int)

print("Flattened X2:", X2.shape)

# ---- train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

# ---- model ----
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

# ---- train ----
rf.fit(X_train, y_train)

# ---- evaluate ----
pred = rf.predict(X_test)
acc = accuracy_score(y_test, pred)
print("\nAccuracy:", acc)

print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification report:\n", classification_report(y_test, pred))

# ---- save ----
dump(rf, model_path)
print("\nSaved RF model to:", model_path)


In [ ]:
#Loading testig dataset and evaluate the model on it
import numpy as np
from joblib import load
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report

rf = load(model_path)
d = np.load(test_set_path)
d = np.load(Path("../datasets/research_random_test.npz"))
X_test = d["dataset"]
y_test = d["label"] if "label" in d.files else None

print("X_test:", X_test.shape)
print("Model expects:", rf.n_features_in_)

# Ensure feature match
if X_test.ndim != 2 or X_test.shape[1] != rf.n_features_in_:
    raise ValueError(f"Feature mismatch: got {X_test.shape}, expected (*, {rf.n_features_in_})")

y_pred = rf.predict(X_test)

if y_test is not None:
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
else:
    print("Predictions (first 20):", y_pred[:20])

In [ ]:
import pandas as pd
drf = pd.DataFrame(rf.feature_importances_, columns=["feature_importance"])

In [ ]:
display(drf)